In [13]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv('/content/diabetes.csv')

In [16]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [17]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.495990
BloodPressure,0.174469
SkinThickness,0.295138
Insulin,0.377081
BMI,0.315577
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [18]:
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [19]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [20]:
x = scaler.fit_transform(x)

In [21]:
x

array([[ 0.63994726,  0.86462486, -0.03218035, ...,  0.16948251,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.20472661, -0.52812374, ..., -0.84854874,
        -0.36506078, -0.19067191],
       [ 1.23388019,  2.01426457, -0.69343821, ..., -1.32847775,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 , -0.02224005, -0.03218035, ..., -0.90672195,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.14199419, -1.02406713, ..., -0.33953311,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.94195182, -0.19749482, ..., -0.2959032 ,
        -0.47378505, -0.87137393]])

In [22]:
x.shape

(768, 8)

In [23]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [65]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout

In [25]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
model.fit(x_train,y_train,batch_size=32,epochs=100,validation_data=(x_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.5000 - loss: 0.7255 - val_accuracy: 0.5390 - val_loss: 0.6791
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5765 - loss: 0.6621 - val_accuracy: 0.6169 - val_loss: 0.6272
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6450 - loss: 0.6148 - val_accuracy: 0.6234 - val_loss: 0.5881
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6775 - loss: 0.5779 - val_accuracy: 0.6883 - val_loss: 0.5557
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7036 - loss: 0.5475 - val_accuracy: 0.7143 - val_loss: 0.5318
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7280 - loss: 0.5226 - val_accuracy: 0.7338 - val_loss: 0.5111
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7443 - loss: 0.5027 - val_accuracy: 0.7273 - val_loss: 0.4955
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7590 - loss: 0.4863 - val_accuracy: 0.7532 - 

In [27]:
# How to select appropriate optimizer
# No. of nodes in a layer
# How to select no. of layers
# All in one model

In [28]:
pip install -U keras-tuner

In [29]:
import kerastuner as kt

In [30]:
def build_model(hp):
  model=Sequential()
  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer=hp.Choice('optimizer', values=['Adam','sgd','rmsprop','adadelta'])
  model.compile(optimizer=optimizer, loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [31]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

In [32]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 4 Complete [00h 00m 03s]
val_accuracy: 0.6038960814476013

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 21s


In [33]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [34]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [35]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 0.7883 - loss: 0.4546 - val_accuracy: 0.8117 - val_loss: 0.4360
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7932 - loss: 0.4406 - val_accuracy: 0.8182 - val_loss: 0.4280
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.7948 - loss: 0.4317 - val_accuracy: 0.8117 - val_loss: 0.4226
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7964 - loss: 0.4238 - val_accuracy: 0.8182 - val_loss: 0.4200
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.7997 - loss: 0.4178 - val_accuracy: 0.8182 - val_loss: 0.4192
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8046 - loss: 0.4120 - val_accuracy: 0.8182 - val_loss: 0.4177
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.8078 - loss: 0.4069 - val_accuracy: 0.8182 - val_loss: 0.4170
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.8013 - loss: 0.4016 - val_accurac

This was about Optimizers.

Now we find out how many neorons should be in a layer ?

In [37]:
def build_model(hp):
  model = Sequential()
  units = hp.Int('units', min_value=8,max_value=128,step=8)
  model.add(Dense(units=units, activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [38]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir',
                        project_name='Salman Khan')

In [39]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.6168830990791321

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 17s


In [40]:
tuner.get_best_hyperparameters()[0].values

{'units': 104}

In [41]:
model = tuner.get_best_models(num_models=1)[0]

In [42]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.7850 - loss: 0.4475
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7915 - loss: 0.4312  
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7948 - loss: 0.4217 
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7997 - loss: 0.4143 
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8013 - loss: 0.4082 
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8029 - loss: 0.4021 
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8094 - loss: 0.3961 
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8078 - loss: 0.3896 
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8176 - loss: 0.3845 
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8225 - loss: 0.3791 
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8274 - loss: 0.3733 
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3m

In [43]:
def build_model(hp):
  model = Sequential()

  model.add(Dense(72,activation='relu',input_dim=8))
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
    model.add(Dense(72,activation='relu'))

  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [44]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='mydir',
                        project_name='num_layers')

In [45]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 3 Complete [00h 00m 06s]
val_accuracy: 0.8181818127632141

Best val_accuracy So Far: 0.850649356842041
Total elapsed time: 00h 00m 17s


In [46]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 6}

Best performance with 7 layers.

In [47]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [48]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 69ms/step - accuracy: 0.8290 - loss: 0.3879 - val_accuracy: 0.8377 - val_loss: 0.4145
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8583 - loss: 0.3521 - val_accuracy: 0.8377 - val_loss: 0.4077
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8648 - loss: 0.3180 - val_accuracy: 0.8247 - val_loss: 0.4760
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8746 - loss: 0.3037 - val_accuracy: 0.8571 - val_loss: 0.4338
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8893 - loss: 0.2705 - val_accuracy: 0.7922 - val_loss: 0.5643
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8974 - loss: 0.2625 - val_accuracy: 0.7662 - val_loss: 0.7633
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8941 - loss: 0.2479 - val_accuracy: 0.8312 - val_loss: 0.4248
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9121 - loss: 0.2277 - val_accuracy: 0.78

In [67]:
def build_model(hp):
    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        if counter == 0:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                    activation=hp.Choice(
                        'activation' + str(i),
                        values=['relu', 'tanh', 'sigmoid']
                    ),
                    input_dim=8
                )
            )

            model.add(
                Dropout(
                    hp.Choice(
                        'dropout' + str(i),
                        values=[0.1, 0.2, 0.3, 0.4, 0.5,
                                0.6, 0.7, 0.8, 0.9]
                    )
                )
            )

        else:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                    activation=hp.Choice(
                        'activation' + str(i),
                        values=['relu', 'tanh', 'sigmoid']
                    )
                )
            )

            model.add(
                Dropout(
                    hp.Choice(
                        'dropout' + str(i),
                        values=[0.1, 0.2, 0.3, 0.4, 0.5,
                                0.6, 0.7, 0.8, 0.9]
                    )
                )
            )

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile
    model.compile(
        optimizer=hp.Choice(
            'optimizer',
            values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [68]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='mydir',
                        project_name='final')

Reloading Tuner from mydir/final/tuner0.json


In [69]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

In [71]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 8,
 'units0': 72,
 'activation0': 'sigmoid',
 'optimizer': 'adadelta',
 'units1': 32,
 'activation1': 'relu',
 'units2': 120,
 'activation2': 'sigmoid',
 'units3': 56,
 'activation3': 'sigmoid',
 'units4': 16,
 'activation4': 'relu',
 'units5': 24,
 'activation5': 'tanh',
 'units6': 128,
 'activation6': 'relu',
 'units7': 8,
 'activation7': 'relu'}

In [72]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adadelta', because it has 2 variables whereas the saved optimizer has 38 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [73]:
model.fit(x_train,y_train,epochs=200,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 14s 268ms/step - accuracy: 0.4837 - loss: 0.6966 - val_accuracy: 0.6429 - val_loss: 0.6898
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5179 - loss: 0.6955 - val_accuracy: 0.6429 - val_loss: 0.6896
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5228 - loss: 0.6925 - val_accuracy: 0.6429 - val_loss: 0.6895
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4853 - loss: 0.6952 - val_accuracy: 0.6429 - val_loss: 0.6893
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5163 - loss: 0.6947 - val_accuracy: 0.6429 - val_loss: 0.6892
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5326 - loss: 0.6927 - val_accuracy: 0.6429 - val_loss: 0.6891
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5212 - loss: 0.6942 - val_accuracy: 0.6429 - val_loss: 0.6889
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5293 - loss: 0.6929 - val_accuracy